In [ ]:
import torch
from torch.optim import AdamW
import numpy as np
import matplotlib.pyplot as plt
from torch import nn
from torch.utils.data import DataLoader
from transformers import BertModel, BertTokenizer, BertPreTrainedModel
from datasets import load_dataset, ClassLabel
from tqdm import tqdm

In [ ]:
class Config:
    bottleneck_dimensions = [8, 32, 64, 128]
    bert_model = "bert-base-uncased"
    num_labels = 150 # latent lables as present in dataset
    dropout_prob = 0.1
    noise_std = 0.1
    batch_size = 32
    bert_lr = 2e-5
    bottleneck_lr = 1e-3
    epochs = 10
    max_seq_length = 136 # max seq length from train dataset

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
def prepare_dataset():
    dataset = load_dataset("Deeppavlov/clinc150")

    print("Sample example:", dataset["train"][0])

    print(f"Train size before filtering: {len(dataset['train'])}")
    print(f"Validation size before filtering: {len(dataset['validation'])}")
    print(f"Test size before filtering: {len(dataset['test'])}")

    # As I saw some label were of NoneType, so I removed them
    def filter_none_labels(example):
        return example["label"] is not None

    dataset = dataset.filter(
        filter_none_labels,
        batched=False
    )

    print(f"Train size after filtering: {len(dataset['train'])}")
    print(f"Validation size after filtering: {len(dataset['validation'])}")
    print(f"Test size after filtering: {len(dataset['test'])}")

    # max_sentence_length = 0
    # for example in dataset['train']:
    #     if len(example['utterance']) > max_sentence_length:
    #         max_sentence_length = len(example['utterance'])
    # print(f"Max sentence length in dataset['train']: {max_sentence_length}")


    tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

    def tokenize_fn(examples):
        return tokenizer(
            examples["utterance"],
            padding="max_length",
            truncation=True,
            max_length=Config.max_seq_length,
            return_tensors="pt"
        )

    tokenized_datasets = dataset.map(
        tokenize_fn,
        batched=True,
        remove_columns=["utterance"]
    )

    tokenized_datasets.set_format("torch", columns=["input_ids", "token_type_ids", "attention_mask", "label"])

    return (
        tokenized_datasets["train"],
        tokenized_datasets["validation"],
        tokenized_datasets["test"]
    )

train_data, validation_data, test_data = prepare_dataset()

def create_dataloader(dataset, batch_size = 32):
    return DataLoader(
        dataset,
        batch_size = batch_size,
        shuffle = True,
        num_workers = 2
    )

In [ ]:
# Part 1 (Bottleneck without Reconstruction)


class BertBottleneckClassifier(nn.Module):
    def __init__(self, bottleneck_dimension):
        super().__init__()
        self.bert = BertModel.from_pretrained(Config.bert_model)
        self.bottleneck = nn.Linear(self.bert.config.hidden_size, bottleneck_dimension)
        self.dropout = nn.Dropout(Config.dropout_prob)
        self.classifier = nn.Linear(bottleneck_dimension, Config.num_labels)

        # Freeze BERT parameters initially
        for param in self.bert.parameters():
            param.requires_grad = False

    def forward(self, input_ids, attention_mask, token_type_ids = None):
        bert_out = self.bert(
            input_ids=input_ids,
            attention_mask = attention_mask,
            token_type_ids = token_type_ids
        )
        cls_embedding = bert_out.last_hidden_state[:, 0, :]
        compressed = self.bottleneck(cls_embedding)
        compressed = self.dropout(compressed)
        return self.classifier(compressed)


def train_model(bottleneck_dimension):
    model = BertBottleneckClassifier(bottleneck_dimension).to(device)

    optimizer = AdamW([
        {"params": model.bert.parameters(), "lr": Config.bert_lr},
        {"params": model.bottleneck.parameters(), "lr": Config.bottleneck_lr},
        {"params": model.classifier.parameters(), "lr": Config.bottleneck_lr}
    ])

    train_loader = create_dataloader(train_data, Config.batch_size)
    val_loader = create_dataloader(validation_data, Config.batch_size)


    best_val_acc = 0
    best_model = None

    for epoch in range(Config.epochs):
        model.train()
        total_loss = 0
        for batch in tqdm(train_loader, desc = f"Training X = {bottleneck_dimension} ----> Epoch = {epoch + 1}"):
            inputs = {k: v.to(device) for k, v in batch.items() if k != "label"}
            labels = batch["label"].to(device)

            optimizer.zero_grad()
            outputs = model(**inputs)
            classification_loss = nn.CrossEntropyLoss()(outputs, labels)
            classification_loss.backward()
            optimizer.step()

            total_loss += classification_loss.item()

        # Model evaluation on validation dataset
        model.eval()
        correct = 0
        total = 0
        with torch.no_grad():
            for batch in val_loader:
                inputs = {k: v.to(device) for k, v in batch.items() if k != "label"}
                labels = batch["label"].to(device)

                outputs = model(**inputs)
                _, predicted = torch.max(outputs, 1)
                total += labels.size(0)
                correct += (predicted == labels).sum().item()

        val_acc = correct / total
        print(f"Epoch = {epoch+1}: Val Acc = {val_acc:.4f}")

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_model = model.state_dict()

    # Load best model
    model.load_state_dict(best_model)
    return model

def test_model(model):
    test_loader = create_dataloader(test_data, Config.batch_size)
    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for batch in test_loader:
            inputs = {k: v.to(device) for k, v in batch.items() if k != "label"}
            labels = batch["label"].to(device)

            outputs = model(**inputs)
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    return correct / total

results = {}

for dim in Config.bottleneck_dimensions:
    print(f"\n{'-'*50} Bottleneck dimension: {dim} {'-'*50}\n")

    model = train_model(dim)
    test_acc = test_model(model)
    results[dim] = test_acc
    print(f"Test Accuracy for X={dim}: {test_acc:.4f}")


dims = sorted(results.keys())
part1_accs = [results[d] for d in dims]

plt.figure(figsize=(10, 6))
plt.plot(dims, part1_accs, 'bo-')
plt.xscale('log', base=2)
plt.xticks(dims, labels = dims)
plt.xlabel('Bottleneck Dimension (X)')
plt.ylabel('Test Accuracy')
plt.title('Classification Accuracy vs Bottleneck Width (Bottleneck without Reconstruction)')
plt.grid(True)
info_text = f"Epochs: {Config.epochs}\nBatch Size: {Config.batch_size}\nTraining Pairs: {len(train_data)}\nValidation Pairs: {len(validation_data)}\nTest Pairs: {len(test_data)}"
plt.text(0.05, 0.95, info_text, transform=plt.gca().transAxes, fontsize=10, verticalalignment='top', bbox=dict(boxstyle='round', facecolor='white', alpha=0.5))
plt.show()

In [ ]:
# Part 2 (Bottleneck with Reconstruction)

class BertBottleneckAutoencoderClassifier(nn.Module):
    def __init__(self, bottleneck_dimension):
        super().__init__()
        self.bert = BertModel.from_pretrained(Config.bert_model)

        self.encoder = nn.Sequential(
            nn.Linear(768, 384),
            nn.ReLU(),
            nn.Linear(384, bottleneck_dimension)
        )

        self.decoder = nn.Sequential(
            nn.Linear(bottleneck_dimension, 384),
            nn.ReLU(),
            nn.Linear(384, 768)
        )

        self.classifier = nn.Sequential(
            # nn.Dropout(Config.dropout_prob),
            nn.Linear(bottleneck_dimension, Config.num_labels)
        )

        for param in self.bert.parameters():
            param.requires_grad = False

    def forward(self, input_ids, attention_mask, token_type_ids = None):
        bert_out = self.bert(input_ids, attention_mask, token_type_ids)
        cls_embedding = bert_out.last_hidden_state[:, 0, :]

        compressed = self.encoder(cls_embedding)
        reconstructed = self.decoder(compressed)

        return {
            'logits': self.classifier(compressed),
            'compressed': compressed,
            'reconstructed': reconstructed,
            'original': cls_embedding
        }

def train_autoencoder(bottleneck_dimension):
    model = BertBottleneckAutoencoderClassifier(bottleneck_dimension).to(device)

    optimizer = AdamW([
        {'params': model.encoder.parameters(), 'lr': Config.bottleneck_lr},
        {'params': model.decoder.parameters(), 'lr': Config.bottleneck_lr},
        {'params': model.classifier.parameters(), 'lr': Config.bottleneck_lr}
    ])

    train_loader = create_dataloader(train_data, Config.batch_size)
    val_loader = create_dataloader(validation_data, Config.batch_size)

    best_val_acc = 0
    best_model = None

    for epoch in range(Config.epochs):
        model.train()
        for batch in tqdm(train_loader, desc = f"Training X = {bottleneck_dimension} ----> Epoch = {epoch + 1}"):
            inputs = {k: v.to(device) for k, v in batch.items() if k != "label"}
            labels = batch['label'].to(device)

            outputs = model(**inputs)
            classification_loss = nn.CrossEntropyLoss()(outputs['logits'], labels)
            # As said in the question reconstruction loss between autoencoder input (i.e. CLS embedding) and final output (i.e. reconstructed output)
            reconstruction_loss = nn.MSELoss()(outputs['reconstructed'], outputs['original'])
            alpha = 1.0  # Fixed classification weight
            beta = 0.5 if bottleneck_dimensions >= 64 else 1.0  # Reduced reconstruction weight
            total_loss = alpha * classification_loss + beta * reconstruction_loss


            optimizer.zero_grad()
            total_loss.backward()
            optimizer.step()

        # Model evaluation on validation dataset
        model.eval()
        correct = 0
        total = 0
        with torch.no_grad():
            for batch in val_loader:
              inputs = {k: v.to(device) for k, v in batch.items() if k != "label"}
              labels = batch['label'].to(device)

              outputs = model(**inputs)
              _, predicted = torch.max(outputs['logits'], 1)
              total += labels.size(0)
              correct += (predicted == labels).sum().item()

        val_acc = correct / total
        print(f"Epoch = {epoch+1}: Val Acc = {val_acc:.4f}")

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_model = model.state_dict()

    # Load best model
    model.load_state_dict(best_model)
    return model

def test_model(model):
    test_loader = create_dataloader(test_data, Config.batch_size)
    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for batch in test_loader:
            inputs = {k: v.to(device) for k, v in batch.items() if k != "label"}
            labels = batch["label"].to(device)

            outputs = model(**inputs)
            _, predicted = torch.max(outputs['logits'], 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    return correct / total

results = {}

for dim in Config.bottleneck_dimensions:
    print(f"\n{'-'*50} Bottleneck dimension: {dim} {'-'*50}\n")

    model = train_autoencoder(dim)
    test_acc = test_model(model)
    results[dim] = test_acc
    print(f'Test Accuracy for X={dim}: {test_acc:.4f}')

dims = sorted(results.keys())
part2_accs = [results[d] for d in dims]

plt.figure(figsize=(10, 6))
plt.plot(dims, part2_accs, 'bo-')
plt.xscale('log', base = 2)
plt.xticks(dims, labels = dims)
plt.xlabel('Bottleneck Dimension (X)')
plt.ylabel('Test Accuracy')
plt.title('Classification Accuracy vs Bottleneck Width (Bot!tleneck with Reconstruction)')
plt.legend()
plt.grid(True)
info_text = f"Epochs: {Config.epochs}\nBatch Size: {Config.batch_size}\nTraining Pairs: {len(train_data)}\nValidation Pairs: {len(validation_data)}\nTest Pairs: {len(test_data)}"
plt.text(0.05, 0.95, info_text, transform=plt.gca().transAxes, fontsize=10, verticalalignment='top', bbox=dict(boxstyle='round', facecolor='white', alpha=0.5))
plt.show()

In [ ]:
# Part 3 (Stochasticity and Information Retention)

class BertBottleneckAutoencoderClassifierNoisy(nn.Module):
    def __init__(self, bottleneck_dimension):
        super().__init__()
        self.bert = BertModel.from_pretrained(Config.bert_model)
        self.bottleneck_dimension = bottleneck_dimension

        self.encoder = nn.Sequential(
            nn.Linear(768, 384),
            nn.ReLU(),
            nn.Linear(384, bottleneck_dimension)
        )

        self.decoder = nn.Sequential(
            nn.Linear(bottleneck_dimension, 384),
            nn.ReLU(),
            nn.Linear(384, 768)
        )

        self.classifier = nn.Sequential(
            # nn.Dropout(Config.dropout_prob),
            nn.Linear(bottleneck_dimension, Config.num_labels)
        )

        for param in self.bert.parameters():
            param.requires_grad = False

    def forward(self, input_ids, attention_mask, token_type_ids=None):
        bert_out = self.bert(input_ids, attention_mask, token_type_ids)
        cls_embedding = bert_out.last_hidden_state[:, 0, :]

        compressed = self.encoder(cls_embedding)

        if self.training:
            effective_std = Config.noise_std / torch.sqrt(torch.tensor(self.bottleneck_dim, dtype=torch.float32))
            noise = torch.randn_like(compressed) * effective_std.to(compressed.device)
            compressed = compressed + noise

        reconstructed = self.decoder(compressed)

        return {
            'logits': self.classifier(compressed),
            'compressed': compressed,
            'reconstructed': reconstructed,
            'original': cls_embedding
        }

def train_autoencoder_noisy(bottleneck_dimension):
    model = BertBottleneckAutoencoderClassifierNoisy(bottleneck_dimension).to(device)

    optimizer = AdamW([
        {'params': model.encoder.parameters(), 'lr': Config.bottleneck_lr},
        {'params': model.decoder.parameters(), 'lr': Config.bottleneck_lr},
        {'params': model.classifier.parameters(), 'lr': Config.bottleneck_lr}
    ])

    train_loader = create_dataloader(train_data, Config.batch_size)
    val_loader = create_dataloader(validation_data, Config.batch_size)

    best_val_acc = 0
    best_model = None

    for epoch in range(Config.epochs):
        model.train()
        for batch in tqdm(train_loader, desc=f"Training X = {bottleneck_dimension} ----> Epoch = {epoch+1}"):
            inputs = {k: v.to(device) for k, v in batch.items() if k != "label"}
            labels = batch["label"].to(device)

            outputs = model(**inputs)
            classification_loss = nn.CrossEntropyLoss()(outputs['logits'], labels)
            # As said in the question reconstruction loss between autoencoder input (i.e. CLS embedding) and final output (i.e. reconstructed output)
            reconstruction_loss = nn.MSELoss()(outputs['reconstructed'], outputs['original'])
            total_loss = classification_loss + reconstruction_loss

            optimizer.zero_grad()
            total_loss.backward()
            optimizer.step()

        # Model evaluation on vaidation dataset
        model.eval()
        correct = 0
        total = 0
        with torch.no_grad():
            for batch in val_loader:
                inputs = {k: v.to(device) for k, v in batch.items() if k != "label"}
                labels = batch["label"].to(device)

                outputs = model(**inputs)
                _, predicted = torch.max(outputs['logits'], 1)
                total += labels.size(0)
                correct += (predicted == labels).sum().item()

        val_acc = correct / total
        print(f"Epoch = {epoch+1}: Val Acc = {val_acc:.5f}")

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_model = model.state_dict()

    # Load best model
    model.load_state_dict(best_model)
    return model

def test_model(model):
    test_loader = create_dataloader(test_data, Config.batch_size)
    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for batch in test_loader:
            inputs = {k: v.to(device) for k, v in batch.items() if k != "label"}
            labels = batch["label"].to(device)

            outputs = model(**inputs)
            _, predicted = torch.max(outputs['logits'], 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    return correct / total

results = {}
for dim in Config.bottleneck_dimensions:
    print(f"\n{'-'*50} Bottleneck dimension: {dim} {'-'*50}")

    model = train_autoencoder_noisy(dim)
    test_acc = test_model(model)
    results[dim] = test_acc
    print(f"Test Accuracy for X={dim}: {test_acc:.4f}")

dims = sorted(results.keys())
part3_accs = [results[d] for d in dims]

plt.figure(figsize=(10, 6))
plt.plot(dims, part3_accs, 'go--')
plt.xscale('log', base = 2)
plt.xticks(dims, labels = dims)
plt.xlabel('Bottleneck Dimension (X)')
plt.ylabel('Test Accuracy')
plt.title('Classification Accuracy vs Bottleneck Width (Stochasticity and Information Retention)')
plt.legend()
plt.grid(True)
info_text = f"Epochs: {Config.epochs}\nBatch Size: {Config.batch_size}\nTraining Pairs: {len(train_data)}\nValidation Pairs: {len(validation_data)}\nTest Pairs: {len(test_data)}"
plt.text(0.05, 0.95, info_text, transform=plt.gca().transAxes, fontsize=10, verticalalignment='top', bbox=dict(boxstyle='round', facecolor='white', alpha=0.5))
plt.show()

In [ ]:
plt.figure(figsize=(10, 6))

plt.plot(dims, part1_accs, 'ro-', label='Part 1 Accuracy')
plt.plot(dims, part2_accs, 'bo-', label='Part 2 Accuracy')
plt.plot(dims, part2_accs, 'go-', label='Part 3 Accuracy')


plt.xscale('log', base=2)
plt.xticks(dims, labels=dims)
plt.xlabel('Bottleneck Dimension (X)')
plt.ylabel('Test Accuracy')
plt.title('Classification Accuracy vs Bottleneck Width')
plt.legend()
plt.grid(True)
info_text = f"Epochs: {Config.epochs}\nBatch Size: {Config.batch_size}\nTraining Pairs: {len(train_data)}\nValidation Pairs: {len(validation_data)}\nTest Pairs: {len(test_data)}\nLearning Rate: {Config.bottleneck_lr}"
plt.text(0.05, 0.95, info_text, transform=plt.gca().transAxes, fontsize=10, verticalalignment='top', bbox=dict(boxstyle='round', facecolor='white', alpha=0.5))
plt.show()


'''
In Part 2, reason why I think for larger bottleneck width the model is not performing good
  - model might retain more information, but some of it could be noise or irrelevant details that gives less accuracy while classification.
  - In part 1 even though the bottleneck width is large, the model was free to keep whatever details it want, but in part2 as we are forcing the model to retain more input details.
    If the bottleneck is wide enough to hold both useful and irrelevant information, the classifier might overfit.

'''